# 02 — Baselines zero-shot (decisão F0.5 do REPLAN)

Clona a voz do Pedro **sem treinar nada** em 3 sistemas license-clean e mede:
**A)** Chatterbox-Multilingual-**pt-br** (MIT, pack dedicado) · **B)** Pocket-TTS-pt
(Kyutai, CC-BY, CPU) · **C)** CSM-1B in-context (Apache). APIs verificadas 2026-06-10.

**Entrada:** 1-3 clipes limpos da voz (do piloto G0, ou grave 10s no celular pra começar).
**GPU:** T4 basta (Pocket roda em CPU). ⚠️ Rode a seção C em runtime SEPARADO
(pins de transformers conflitam com o chatterbox).

**Gate F0.5:** melhor (spk-sim × WER × escuta pt-BR-vs-pt-PT) vira o candidato #1 de finetune.
Se nenhum ≥0.60 spk-sim com pt-BR aceitável → direto pro finetune (notebooks 1→2).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr

# referência de voz: ~7-10s limpos (chatterbox trunca em 10s/6s; mais que isso é descartado)
# sugestão pronta (import ElevenLabs no Mac): data/raw/elevenlabs2024/segments/found_08_seg019.wav
# → copie pro Drive como ref_pedro.wav (ou escolha outro segmento limpo de 7-10s)
REF_WAV = '/content/drive/MyDrive/TTS-ptbr-data/ref_pedro.wav'   # ajuste o caminho
import pathlib; assert pathlib.Path(REF_WAV).exists(), 'suba um wav de referência no Drive'
import json
bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
print(len(bench), 'frases do benchmark congelado')

## A. Chatterbox-Multilingual-pt-br (pack dedicado — o multilingual amplo soa pt-PT, issue #281)

⚠️ **RUNTIME SEPARADO obrigatório.** O `chatterbox-tts` fixa `transformers==5.2.0` +
`torch==2.6` — instalar isso faz upgrade/downgrade pesado do runtime e é
**incompatível com o CSM (seção C, que exige transformers 4.52.3)**. Fluxo:
**runtime 1** = seções A + B + D (Chatterbox/Pocket/eval); **runtime 2 (novo)** =
seção C (CSM). Os áudios `gen_*/` ficam no Drive e a seção D lê de lá.
Após o `pip install chatterbox-tts`, **reinicie o runtime** (a célula força isso) e
re-rode da montagem do Drive.

In [ ]:
!pip -q install chatterbox-tts soundfile jiwer librosa
from huggingface_hub import hf_hub_download
import shutil, pathlib

d = pathlib.Path('/content/ckpt_ptbr'); d.mkdir(exist_ok=True)
for f in ['t3_pt_br.safetensors', 's3gen_v3.pt', 'grapheme_mtl_merged_expanded_v1.json']:
    shutil.copy(hf_hub_download('ResembleAI/Chatterbox-Multilingual-pt-br', f), d / f)
for f in ['ve.pt', 'conds.pt', 'Cangjie5_TC.json']:    # voice encoder vem do repo base
    shutil.copy(hf_hub_download('ResembleAI/chatterbox', f), d / f)
if (d / 's3gen_v3.pt').exists():
    (d / 's3gen_v3.pt').rename(d / 's3gen.pt')          # from_local carrega 's3gen.pt'

import torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_local(d, device='cuda', t3_model='t3_pt_br.safetensors')

out = pathlib.Path('gen_chatterbox'); out.mkdir(exist_ok=True)
for i, item in enumerate(bench):
    wav = model.generate(item['text'], language_id='pt', audio_prompt_path=REF_WAV,
                         exaggeration=0.5, cfg_weight=0.5, temperature=0.8)
    ta.save(str(out / f"{item.get('id', i):0>3}.wav"), wav, model.sr)   # 24kHz; watermark Perth embutida
print('✅ chatterbox pt-br ok →', out)
# expressivo: exaggeration~0.7 + cfg_weight~0.3 | referência com fala rápida: cfg_weight~0.3

## B. Pocket-TTS português (CPU, ~200ms; clone exige aceitar o gate em hf.co/kyutai/pocket-tts)

In [ ]:
!pip -q install pocket-tts
import os, json, pathlib, subprocess
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# guard: o modelo de clone é GATED — a conta do HF_TOKEN PRECISA ter aceito os termos
# em https://huggingface.co/kyutai/pocket-tts (senão cai no fallback sem clonagem)
from huggingface_hub import HfApi
try:
    HfApi().model_info('kyutai/pocket-tts', token=os.environ['HF_TOKEN'])
    print('✅ acesso ao pocket-tts (clone) ok')
except Exception as e:
    print('⚠️ sem acesso ao kyutai/pocket-tts — aceite os termos no HF com a conta do seu token:', e)

# pt só existe na variante 24 camadas (NÃO há 'portuguese' 6L) — docs kyutai pocket-tts
!pocket-tts export-voice {REF_WAV} /content/pedro_voice.safetensors --language portuguese_24l
for variant in ['portuguese_24l']:
    out = pathlib.Path(f'gen_pocket_{variant}'); out.mkdir(exist_ok=True)
    for i, item in enumerate(bench):
        subprocess.run(['pocket-tts', 'generate', '--language', variant,
                        '--voice', '/content/pedro_voice.safetensors',
                        '--text', item['text'],
                        '--output-path', str(out / f"{item.get('id', i):0>3}.wav")], check=True)
    print('✅', variant, '→', out)
# escute: o clone soa pt-BR ou pt-PT? ANOTE — é decisivo pro gate F0.5.

## C. CSM-1B in-context (⚠️ RUNTIME NOVO: pins conflitam) 

In [ ]:
# ⚠️ RODE ISTO EM RUNTIME NOVO (transformers 4.52.3 conflita com o chatterbox da seção A)
# !pip -q install "transformers==4.52.3" soundfile librosa torchcodec && pip uninstall -y torchao
# from google.colab import drive, userdata; drive.mount('/content/drive')
# import torch, soundfile as sf, librosa, json, pathlib
# from transformers import CsmForConditionalGeneration, AutoProcessor
# REF_WAV='/content/drive/MyDrive/TTS-ptbr-data/ref_pedro.wav'
# bench=[json.loads(l) for l in open('eval/benchmark_ptbr.jsonl',encoding='utf-8') if l.strip()]
# proc = AutoProcessor.from_pretrained('unsloth/csm-1b')
# csm, info = CsmForConditionalGeneration.from_pretrained('unsloth/csm-1b',
#     torch_dtype=torch.bfloat16, output_loading_info=True)
# assert not [k for k in info.get('missing_keys',[]) if 'embed_audio' in k], 'pesos de áudio faltando — versão de transformers errada'
# csm = csm.to('cuda')
# ref_arr, _ = librosa.load(REF_WAV, sr=24000, mono=True)
# REF_TEXT = 'TRANSCRICAO EXATA DO SEU WAV DE REFERENCIA AQUI'
# out = pathlib.Path('gen_csm_zeroshot'); out.mkdir(exist_ok=True)
# for i, item in enumerate(bench):
#     conv = [{'role':'0','content':[{'type':'text','text':REF_TEXT},{'type':'audio','path':ref_arr}]},
#             {'role':'0','content':[{'type':'text','text':item['text']}]}]
#     inputs = proc.apply_chat_template(conv, tokenize=True, return_dict=True)
#     audio = csm.generate(**inputs.to('cuda'), max_new_tokens=375, output_audio=True)
#     sf.write(str(out / f"{item.get('id', i):0>3}.wav"), audio[0].to(torch.float32).cpu().numpy(), 24000)
# !cp -r gen_csm_zeroshot /content/drive/MyDrive/TTS-ptbr-data/   # leva pro Drive p/ a seção D
# print('✅ csm zero-shot → gen_csm_zeroshot (no Drive)')

## D. Eval comparativa (spk-sim + WER) → tabela de decisão

In [ ]:
!pip -q install faster-whisper==1.1.0
import os, pathlib
# cuDNN9 path p/ faster-whisper não crashar (mesmo fix do nb0)
try:
    import nvidia.cudnn
    os.environ['LD_LIBRARY_PATH'] = str(pathlib.Path(nvidia.cudnn.__file__).parent/'lib')+':'+os.environ.get('LD_LIBRARY_PATH','')
except Exception: pass
!mkdir -p ref_pedro && cp {REF_WAV} ref_pedro/
for system in ['gen_chatterbox', 'gen_pocket_portuguese_24l', 'gen_csm_zeroshot']:
    if not pathlib.Path(system).exists(): continue
    print(f'===== {system} =====')
    !python -m eval.wer_roundtrip --in-dir {system} --transcripts eval/benchmark_ptbr.jsonl --model medium --lang pt
    !python -m eval.speaker_sim --ref-dir ref_pedro --gen-dir {system}

## E. Decisão (preencher e levar pro REPLAN)

| Sistema | spk-sim | WER | Soa pt-BR? (ouvido) | Emoção controlável? | Veredito |
|---|---|---|---|---|---|
| Chatterbox-pt-br | | | | exaggeration/cfg | |
| Pocket-pt (24L) | | | | não | |
| CSM zero-shot | | | | contexto | |

→ Gate: melhor sistema = candidato #1; espelhe os `gen_*/` no Drive pra escuta
comparativa, e registre o resultado em `research/VIGIL-LOG.md` + REPLAN §F0.5.
(Pocket pt só tem a variante 24L — não existe 6L destilada em português.)